In [ ]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

DATA_DIR = Path("data")
input_file = DATA_DIR / "nano33ble_sensor_capture.csv"

FEATURE_COL = "avg_accel_x"
LABEL_COL = "Label"

df = pd.read_csv(input_file)
df[FEATURE_COL] = pd.to_numeric(df[FEATURE_COL], errors="coerce")

X_data = df[[FEATURE_COL]].to_numpy()  # Shaped as (N, 1) for the network
y_data = df[LABEL_COL].map({"Y": 1, "N": 0}).to_numpy() # Shape (N,) for binary classification and ensure it's numeric

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.20, random_state=42)

# Calculate normalization constants from training data
X_mean = X_train.mean()
X_std = X_train.std()

# Manually scale the training and testing sets
X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

print(f"Scaling Parameters: Mean: {X_mean}, Std: {X_std}")
print(f"Training samples: {len(X_train)} | Testing samples: {len(X_test)}")

In [ ]:

model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(1,)),  # Explicitly declare input shape for TFLite
        tf.keras.layers.Dense(8, activation="relu"),  
        tf.keras.layers.Dense(1, activation="sigmoid"),  # Sigmoid activation since this is a binary classification problem
    ]
)

model.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)

model.fit(X_train_scaled, y_train, 
    epochs=20, batch_size=16, 
    validation_data=(X_test_scaled, y_test)
)


In [ ]:
import pathlib

model_dir = pathlib.Path("./models")
model_dir.mkdir(parents=True, exist_ok=True)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]    

tflite_model = converter.convert()

tflite_model_file = model_dir/'model.tflite'     
tflite_model_file.write_bytes(tflite_model)